In [3]:
"""PyCharm (No LangChain) — Stability AI Text-to-Image (minimal, our style)"""

# ============================================================================
# IMPORTS
# ============================================================================
import os, re, base64, datetime, requests
from pathlib import Path
from dotenv import load_dotenv

In [4]:
# ============================================================================
# ENVIRONMENT SETUP (load once at module level, fail-fast)
# ============================================================================
load_dotenv()
STABILITY_API_KEY = os.getenv("STABILITY_API_KEY")
if not STABILITY_API_KEY:
    raise RuntimeError("Missing STABILITY_API_KEY in .env")

API_HOST  = os.getenv("STABILITY_API_HOST") or "https://api.stability.ai"
ENGINE_ID = os.getenv("STABILITY_ENGINE_ID") or "stable-diffusion-xl-1024-v1-0"

In [5]:

# ============================================================================
# HELPERS
# ============================================================================
def safe_name(s: str) -> str:
    s = re.sub(r"[^\w_.)( -]", "", s).strip()
    return re.sub(r"\s+", "_", s)

def save_base64_png(b64: str, out_dir: Path, stem: str) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fp = out_dir / f"sd_{safe_name(stem)}_{ts}.png"
    fp.write_bytes(base64.b64decode(b64))
    return fp

In [6]:
# ============================================================================
# MAIN
# ============================================================================
def main():
    prompt = "A panda surfing on a wave"  # ← เปลี่ยนข้อความได้ตามต้องการ
    out_dir = Path("images")

    resp = requests.post(
        f"{API_HOST}/v1/generation/{ENGINE_ID}/text-to-image",
        headers={
            "Authorization": f"Bearer {STABILITY_API_KEY}",
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        json={
            "text_prompts": [{"text": prompt}],
            "cfg_scale": 7,
            "height": 1024,
            "width": 1024,
            "samples": 1,
            "steps": 50,
        },
        timeout=60,
    )

    if resp.status_code != 200:
        raise RuntimeError(f"Non-200 response: {resp.status_code} → {resp.text}")

    data = resp.json()
    artifacts = data.get("artifacts", [])
    if not artifacts:
        raise RuntimeError("No artifacts in response")

    # บันทึกภาพแรก (minimal)
    path = save_base64_png(artifacts[0]["base64"], out_dir, prompt)
    print(f"Saved: {path}")

# ============================================================================
# SCRIPT EXECUTION
# ============================================================================
if __name__ == "__main__":
    main()

Saved: images\sd_A_panda_surfing_on_a_wave_20251018_163501.png
